# Baseline pointwise metrics

Computes filtered pointwise metrics for baseline accuracy-test runs, reusing the
exact functions the ablation merge pipeline uses (`analysis/filtering.py` and
`ablation/merge_ablation_runs.py`), so numbers are directly comparable.

"Filtered" = instances whose paper is in `paper_blocklist.yaml` are dropped, plus
the pointwise partners of those papers (the same collateral exclusion the pairwise
side does).

Add ai-scientist runs to `RUNS` below when they exist — nothing else changes.

In [24]:
import sys
from pathlib import Path

import pandas as pd

ROOT = next(
    p for p in [Path.cwd(), *Path.cwd().parents]
    if (p / "src").is_dir() and (p / "output").is_dir()
)
sys.path.insert(0, str(ROOT / "src"))

from novelty_eval.analysis.artifacts import (
    _extract_instances_path,
    _extract_mode_from_artifact_dir,
)
from novelty_eval.analysis.filtering import (
    _load_paper_blocklist,
    _derive_exclude_indices,
    _recompute_pointwise_metrics,
)
from novelty_eval.ablation.merge_ablation_runs import (
    _derive_pointwise_partner_excluded,
)

BLOCKED_TITLES = _load_paper_blocklist()
print(f"{len(BLOCKED_TITLES)} blocked titles")

13 blocked titles


In [25]:
# (baseline, setting, model, artifact_dir)
RUNS = [
    ("scideator", "hvh", "gpt-5.4", "output/accuracy_test_artifacts/2026-08-20_12-42-17"),
    ("scideator", "hvh", "opus-4-6", "output/accuracy_test_artifacts/2026-08-20_17-55-09"),
    ("scideator", "vanilla", "gpt-5.4", "output/accuracy_test_artifacts/2026-08-21_10-11-44"),
    ("scideator", "vanilla", "opus-4-6", "output/accuracy_test_artifacts/2026-08-21_15-14-20"),
    ("ai_scientist", "hvh", "gpt-5.4", "output/accuracy_test_artifacts/2026-08-22_10-22-25"),
    ("ai_scientist", "hvh", "opus-4-6", "output/accuracy_test_artifacts/2026-08-22_12-54-19"),
    ("ai_scientist", "vanilla", "gpt-5.4",  "output/accuracy_test_artifacts/2026-08-22_15-21-34"),
    ("ai_scientist", "vanilla", "opus-4-6",  "output/accuracy_test_artifacts/2026-08-22_16-27-02")
]

In [26]:
def filtered_pointwise_metrics(artifact_dir: str) -> dict:
    """Recompute pointwise metrics on the blocklist-filtered subset of one artifact dir."""
    d = (ROOT / artifact_dir).resolve()
    mode = _extract_mode_from_artifact_dir(d)
    if mode != "pointwise":
        raise ValueError(f"{d.name}: expected a pointwise run, got mode={mode!r}")

    instances_path = _extract_instances_path(d)
    excluded = {
        **_derive_exclude_indices(instances_path, BLOCKED_TITLES),
        **_derive_pointwise_partner_excluded(instances_path, BLOCKED_TITLES),
    }
    metrics = _recompute_pointwise_metrics(d, set(excluded))
    if metrics is None:
        raise ValueError(f"{d.name}: no pointwise scores.json found")

    return {
        "run_dir": d.name,
        "instances": instances_path,
        "n_runs": len(list(d.glob("run_pointwise_*/scores.json"))),
        "n_excluded": len(excluded),
        "support": metrics["support"],
        "accuracy": metrics["accuracy"],
        "f1_macro": metrics["f1_macro"],
        "f1_pos": metrics["f1_pos"],
        "f1_neg": metrics["f1_neg"],
        "precision_pos": metrics["precision_pos"],
        "precision_neg": metrics["precision_neg"],
        "recall_pos": metrics["recall_pos"],
        "recall_neg": metrics["recall_neg"],
    }


df = pd.DataFrame(
    {"baseline": baseline, "setting": setting, "model": model, **filtered_pointwise_metrics(path)}
    for baseline, setting, model, path in RUNS
)
df

,baseline,setting,model,run_dir,instances,n_runs,n_excluded,support,accuracy,f1_macro,f1_pos,f1_neg,precision_pos,precision_neg,recall_pos,recall_neg
0,scideator,hvh,gpt-5.4,2026-08-20_12-42-17,output/iclr_test_instances/ablation/pointwise-...,1,14,299,0.588629,0.521488,0.700730,0.342246,0.560311,0.761905,0.935065,0.220690
1,scideator,hvh,opus-4-6,2026-08-20_17-55-09,output/iclr_test_instances/ablation/pointwise-...,1,14,299,0.612040,0.559803,0.711443,0.408163,0.576613,0.784314,0.928571,0.275862
2,scideator,vanilla,gpt-5.4,2026-08-21_10-11-44,output/iclr_test_instances/pairwise_data/20260...,1,14,308,0.581169,0.526092,0.687651,0.364532,0.548263,0.755102,0.922078,0.240260
3,scideator,vanilla,opus-4-6,2026-08-21_15-14-20,output/iclr_test_instances/pairwise_data/20260...,1,14,308,0.662338,0.641652,0.727749,0.555556,0.609649,0.812500,0.902597,0.422078
4,ai_scientist,hvh,gpt-5.4,2026-08-22_10-22-25,output/iclr_test_instances/ablation/pointwise-...,1,14,299,0.548495,0.480749,0.293194,0.668305,0.756757,0.519084,0.181818,0.937931
5,ai_scientist,hvh,opus-4-6,2026-08-22_12-54-19,output/iclr_test_instances/ablation/pointwise-...,1,14,299,0.675585,0.651896,0.742706,0.561086,0.627803,0.815789,0.909091,0.427586
6,ai_scientist,vanilla,gpt-5.4,2026-08-22_15-21-34,output/iclr_test_instances/pairwise_data/20260...,1,14,308,0.574675,0.480741,0.259887,0.701595,1.000000,0.540351,0.149351,1.000000
7,ai_scientist,vanilla,opus-4-6,2026-08-22_16-27-02,output/iclr_test_instances/pairwise_data/20260...,1,14,308,0.775974,0.772324,0.801153,0.743494,0.720207,0.869565,0.902597,0.649351


In [27]:
cols = ["baseline", "model", "f1_macro", "f1_pos", "f1_neg"]

for setting, group in df.groupby("setting", sort=False):
    print(f"=== {setting} ===")
    print(group[cols].to_string(index=False, float_format="%.3f"))
    print()


=== hvh ===
    baseline    model  f1_macro  f1_pos  f1_neg
   scideator  gpt-5.4     0.521   0.701   0.342
   scideator opus-4-6     0.560   0.711   0.408
ai_scientist  gpt-5.4     0.481   0.293   0.668
ai_scientist opus-4-6     0.652   0.743   0.561

=== vanilla ===
    baseline    model  f1_macro  f1_pos  f1_neg
   scideator  gpt-5.4     0.526   0.688   0.365
   scideator opus-4-6     0.642   0.728   0.556
ai_scientist  gpt-5.4     0.481   0.260   0.702
ai_scientist opus-4-6     0.772   0.801   0.743



In [28]:
# (column, header) pairs — the metrics the table reports
METRICS = [
    ("f1_macro", r"\textbf{F$_1$}"),
    ("f1_pos", r"\textbf{F$_1^{+}$}"),
    ("f1_neg", r"\textbf{F$_1^{-}$}"),
]

BASELINE_NAMES = {"scideator": "Scideator", "ai_scientist": "AI-Scientist"}
SETTING_NAMES = {"hvh": "Human vs.\\ Human", "vanilla": "Vanilla"}

CAPTION = "Results for additional, designated novelty evaluation baselines."
LABEL = "tab:baseline_pointwise"


def latex_table(df, metrics=METRICS, caption=CAPTION, label=LABEL) -> str:
    """LaTeX table: a row per (baseline, model), grouped by setting.

    Rows are sorted by baseline name, then by backbone; the best value per column
    is bolded within each setting.
    """
    n_cols = len(metrics) + 1
    header = " & ".join([r"\textbf{Baseline}"] + [h for _, h in metrics])
    lines = [
        r"\begin{table}[t]",
        r"\centering",
        r"\footnotesize",
        r"\setlength{\tabcolsep}{4pt}",
        rf"\begin{{tabular}}{{l{'c' * len(metrics)}}}",
        r"\toprule",
        rf"{header} \\",
    ]
    for i, (setting, group) in enumerate(df.groupby("setting", sort=False)):
        lines.append(r"\midrule" if i == 0 else r"\addlinespace")
        name = SETTING_NAMES.get(setting, setting)
        lines.append(rf"\multicolumn{{{n_cols}}}{{l}}{{\textit{{{name}}}}} \\")
        best = {metric: group[metric].max() for metric, _ in metrics}
        rows = sorted(
            group.itertuples(),
            key=lambda r: (BASELINE_NAMES.get(r.baseline, r.baseline), r.model),
        )
        for row in rows:
            cells = []
            for metric, _ in metrics:
                score = getattr(row, metric)
                value = f"{score:.2f}"
                cells.append(rf"\textbf{{{value}}}" if score == best[metric] else value)
            baseline = BASELINE_NAMES.get(row.baseline, row.baseline)
            lines.append(" & ".join([rf"{baseline} (\texttt{{{row.model}}})"] + cells) + r" \\")
    lines += [
        r"\bottomrule",
        r"\end{tabular}",
        rf"\caption{{{caption}}}",
        rf"\label{{{label}}}",
        r"\end{table}",
    ]
    return "\n".join(lines)


print(latex_table(df))


\begin{table}[t]
\centering
\footnotesize
\setlength{\tabcolsep}{4pt}
\begin{tabular}{lccc}
\toprule
\textbf{Baseline} & \textbf{F$_1$} & \textbf{F$_1^{+}$} & \textbf{F$_1^{-}$} \\
\midrule
\multicolumn{4}{l}{\textit{Human vs.\ Human}} \\
AI-Scientist (\texttt{gpt-5.4}) & 0.48 & 0.29 & \textbf{0.67} \\
AI-Scientist (\texttt{opus-4-6}) & \textbf{0.65} & \textbf{0.74} & 0.56 \\
Scideator (\texttt{gpt-5.4}) & 0.52 & 0.70 & 0.34 \\
Scideator (\texttt{opus-4-6}) & 0.56 & 0.71 & 0.41 \\
\addlinespace
\multicolumn{4}{l}{\textit{Vanilla}} \\
AI-Scientist (\texttt{gpt-5.4}) & 0.48 & 0.26 & 0.70 \\
AI-Scientist (\texttt{opus-4-6}) & \textbf{0.77} & \textbf{0.80} & \textbf{0.74} \\
Scideator (\texttt{gpt-5.4}) & 0.53 & 0.69 & 0.36 \\
Scideator (\texttt{opus-4-6}) & 0.64 & 0.73 & 0.56 \\
\bottomrule
\end{tabular}
\caption{Results for additional, designated novelty evaluation baselines.}
\label{tab:baseline_pointwise}
\end{table}
